In [ ]:
"""
에러가 나면 가장 먼저 이 코드를 실행
!pip install protobuf==3.20.3 --quiet
!pip install --upgrade transformers datasets accelerate --quiet
!pip install evaluate --quiet
"""

In [1]:
!pip install transformers==4.44.2 accelerate==0.33.0 bitsandbytes==0.43.0 peft==0.11.1 datasets==2.19.1 scikit-learn --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 MB 16.0 MB/s eta 0:00:0000:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 55.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 71.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.9 MB/s eta 0:0

In [ ]:
# ====================================================
# 0️⃣ 환경 세팅
# ====================================================
!pip install -q --upgrade transformers peft datasets accelerate tqdm

import os
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ====================================================
# 1️⃣ 데이터 로딩
# ====================================================
train_df = pd.read_csv("/kaggle/input/hallym-nlp-competetion-2025/Train.csv")
test_df  = pd.read_csv("/kaggle/input/hallym-nlp-competetion-2025/Test.csv")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    test_size=0.1,
    stratify=train_df['label'],
    random_state=42
)

# ====================================================
# 2️⃣ Dataset (Text Dropout 포함)
# ====================================================
class NewsDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512, aug_prob=0.1):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.aug_prob = aug_prob

    def text_dropout(self, text):
        words = text.split()
        if len(words) < 3:
            return text
        n_drop = max(1, int(len(words) * self.aug_prob))
        drop_idx = random.sample(range(len(words)), n_drop)
        return " ".join([w for i, w in enumerate(words) if i not in drop_idx])

    def __getitem__(self, idx):
        text = self.texts[idx]
        # 30% 확률로 텍스트 dropout 적용
        if random.random() < 0.3:
            text = self.text_dropout(text)

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.texts)

# ====================================================
# 3️⃣ Tokenizer & Model
# ====================================================
model_name = "EleutherAI/pythia-70m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id

# LoRA 적용
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)
model.to(device)
model.gradient_checkpointing_enable()

# ====================================================
# 4️⃣ Loss, Optimizer, Scheduler
# ====================================================
# Label smoothing 적용
ce_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        cosine_sim = torch.nn.functional.cosine_similarity(
            features.unsqueeze(1), features.unsqueeze(0), dim=-1
        )
        exp_sim = torch.exp(cosine_sim / self.temperature) * mask
        log_prob = torch.log(exp_sim.sum(dim=1) + 1e-8)
        loss = - (torch.log(exp_sim.diagonal() + 1e-8) - log_prob).mean()
        return loss

contrastive_loss = SupConLoss()
lr = 1e-4
optimizer = AdamW(model.parameters(), lr=lr)

# ====================================================
# 5️⃣ DataLoader + Dynamic Padding
# ====================================================
train_dataset = NewsDataset(train_texts, train_labels, tokenizer)
val_dataset   = NewsDataset(val_texts, val_labels, tokenizer)
collator = DataCollatorWithPadding(tokenizer)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collator)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collator)

epochs = 3
accum_steps = 2
total_steps = len(train_loader) * epochs // accum_steps
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# ====================================================
# 6️⃣ Training Loop (FP16 + Accumulation + F1 Log)
# ====================================================
scaler = GradScaler()
best_f1 = 0.0
best_model_path = "best_model.pt"
f1_log = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}", disable=False)

    # epoch별 동적 SupCon 가중치
    supcon_weight = 0.05 * (epoch / epochs + 0.5)

    for i, batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with autocast():
            outputs = model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
            logits = outputs.logits
            hidden = outputs.hidden_states[-1][:, 0, :]
            loss = ce_loss_fn(logits.float(), labels) + supcon_weight * contrastive_loss(hidden, labels)
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * accum_steps
        if (i + 1) % 100 == 0:
            avg_loss = total_loss / (i + 1)
            loop.set_postfix(loss=avg_loss)

    # ===========================
    # Validation
    # ===========================
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs.logits, dim=1)
            preds.extend(pred.cpu().numpy())
            true.extend(labels.cpu().numpy())

    f1 = f1_score(true, preds)
    f1_log.append(f1)
    print(f"\n✅ Epoch {epoch+1} Validation F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_model_path)
        print(f"💾 Saved Best Model (F1={best_f1:.4f})")

# Epoch별 점수 출력
print("\n📊 Epoch별 Validation F1 점수:")
for i, score in enumerate(f1_log, 1):
    mark = "✅ Best" if score == best_f1 else ""
    print(f"  {i}: {score:.4f} {mark}")

# ====================================================
# 7️⃣ Test & Submission
# ====================================================
model.load_state_dict(torch.load(best_model_path))
model.eval()

test_dataset = NewsDataset(test_df['text'].tolist(), tokenizer=tokenizer)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collator)

submission_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting", total=len(test_loader), disable=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1)
        submission_preds.extend(pred.cpu().numpy())

submission = pd.DataFrame({"id": test_df['id'], "label": submission_preds})
submission.to_csv("submission1.csv", index=False)
print("\n🎯 Submission saved as submission.csv")


모델: EleutherAI/pythia-70m (작은 GPT류 언어모델)

기법:

LoRA(경량 파인튜닝)

FP16 혼합정밀 학습 (GradScaler + autocast)

Label Smoothing

Supervised Contrastive Loss (SupConLoss)

Text Dropout (텍스트 단어 일부 제거)

Gradient Accumulation

Learning Rate Scheduler (Warmup 포함)

F1-score 기반 성능 검증 및 best model 저장

In [1]:
# ====================================================
# 0️⃣ 환경 세팅
# ====================================================
!pip install -q --upgrade transformers peft datasets accelerate tqdm

import os
import random
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import pandas as pd
from tqdm import tqdm
from torch.cuda.amp import GradScaler, autocast

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ====================================================
# 1️⃣ 데이터 로딩
# ====================================================
train_df = pd.read_csv("/kaggle/input/hallym-nlp-competetion-2025/Train.csv")
test_df  = pd.read_csv("/kaggle/input/hallym-nlp-competetion-2025/Test.csv")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    test_size=0.1,
    stratify=train_df['label'],
    random_state=42
)

# ====================================================
# 2️⃣ Dataset (Text Dropout 포함)
# ====================================================
class NewsDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=512, aug_prob=0.1):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.aug_prob = aug_prob

    def text_dropout(self, text):
        words = text.split()
        if len(words) < 3:
            return text
        n_drop = max(1, int(len(words) * self.aug_prob))
        drop_idx = random.sample(range(len(words)), n_drop)
        return " ".join([w for i, w in enumerate(words) if i not in drop_idx])

    def __getitem__(self, idx):
        text = self.texts[idx]
        # 30% 확률로 텍스트 dropout 적용
        if random.random() < 0.3:
            text = self.text_dropout(text)

        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.texts)

# ====================================================
# 3️⃣ Tokenizer & Model
# ====================================================
model_name = "EleutherAI/pythia-70m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id

# LoRA 적용
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
)
model = get_peft_model(model, lora_config)
model.to(device)
model.gradient_checkpointing_enable()

# ====================================================
# 4️⃣ Loss, Optimizer, Scheduler
# ====================================================
# Label smoothing 적용
ce_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.05)

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        device = features.device
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        cosine_sim = torch.nn.functional.cosine_similarity(
            features.unsqueeze(1), features.unsqueeze(0), dim=-1
        )
        exp_sim = torch.exp(cosine_sim / self.temperature) * mask
        log_prob = torch.log(exp_sim.sum(dim=1) + 1e-8)
        loss = - (torch.log(exp_sim.diagonal() + 1e-8) - log_prob).mean()
        return loss

contrastive_loss = SupConLoss()
lr = 1e-4
optimizer = AdamW(model.parameters(), lr=lr)

# ====================================================
# 5️⃣ DataLoader + Dynamic Padding
# ====================================================
train_dataset = NewsDataset(train_texts, train_labels, tokenizer)
val_dataset   = NewsDataset(val_texts, val_labels, tokenizer)
collator = DataCollatorWithPadding(tokenizer)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collator)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collator)

epochs = 3
accum_steps = 2
total_steps = len(train_loader) * epochs // accum_steps
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

# ====================================================
# 6️⃣ Training Loop (FP16 + Accumulation + F1 Log)
# ====================================================
scaler = GradScaler()
best_f1 = 0.0
best_model_path = "best_model.pt"
f1_log = []

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}", disable=False)

    # epoch별 동적 SupCon 가중치
    supcon_weight = 0.05 * (epoch / epochs + 0.5)

    for i, batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with autocast():
            outputs = model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
            logits = outputs.logits
            hidden = outputs.hidden_states[-1][:, 0, :]
            loss = ce_loss_fn(logits.float(), labels) + supcon_weight * contrastive_loss(hidden, labels)
            loss = loss / accum_steps

        scaler.scale(loss).backward()

        if (i + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        total_loss += loss.item() * accum_steps
        if (i + 1) % 100 == 0:
            avg_loss = total_loss / (i + 1)
            loop.set_postfix(loss=avg_loss)

    # ===========================
    # Validation
    # ===========================
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            pred = torch.argmax(outputs.logits, dim=1)
            preds.extend(pred.cpu().numpy())
            true.extend(labels.cpu().numpy())

    f1 = f1_score(true, preds)
    f1_log.append(f1)
    print(f"\n✅ Epoch {epoch+1} Validation F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), best_model_path)
        print(f"💾 Saved Best Model (F1={best_f1:.4f})")

# Epoch별 점수 출력
print("\n📊 Epoch별 Validation F1 점수:")
for i, score in enumerate(f1_log, 1):
    mark = "✅ Best" if score == best_f1 else ""
    print(f"  {i}: {score:.4f} {mark}")

# ====================================================
# 7️⃣ Validation 기반 Best Threshold 탐색
# ====================================================
print("\n🔎 Finding best threshold based on validation F1...")

model.load_state_dict(torch.load(best_model_path))
model.eval()

val_probs, val_true = [], []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Validating for threshold"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1]  # 클래스 1 확률
        val_probs.extend(probs.cpu().numpy())
        val_true.extend(labels.cpu().numpy())

import numpy as np
best_t, best_f1 = 0.5, 0.0
for t in np.arange(0.1, 0.9, 0.01):
    preds = [1 if p >= t else 0 for p in val_probs]
    f1 = f1_score(val_true, preds)
    if f1 > best_f1:
        best_t, best_f1 = t, f1

print(f"✅ Best threshold found: {best_t:.2f} (F1={best_f1:.4f})")

# ====================================================
# 8️⃣ Test & Confidence CSV 저장
# ====================================================
print("\n🚀 Generating confidence scores with best threshold...")
threshold = best_t  # 최적 threshold 사용

test_dataset = NewsDataset(test_df['text'].tolist(), tokenizer=tokenizer)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collator)

confidence_scores = []
final_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting", total=len(test_loader), disable=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1]  # 클래스 1 확률

        mapped_confidence = []
        for p in probs.cpu().numpy():
            # 확률을 확신도로 선형 매핑
            if p <= threshold:
                conf = (p / threshold) * 0.5
            else:
                conf = 0.5 + ((p - threshold) / (1 - threshold)) * 0.5
            mapped_confidence.append(conf)

        confidence_scores.extend(mapped_confidence)
        preds = [1 if c > 0.5 else 0 for c in mapped_confidence]
        final_preds.extend(preds)

# CSV 저장
submission_confidence = pd.DataFrame({
    "id": test_df["id"],
    "text": test_df["text"],
    "confidence": confidence_scores,
    "pred_label": final_preds
})

confidence_path = "confidence_pythia_lora_supcon.csv"
submission_confidence.to_csv(confidence_path, index=False)
print(f"\n🎯 Confidence CSV saved as: {confidence_path}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 120.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 41.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 83.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211

2025-12-07 07:03:34.761076: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765091014.939428      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765091014.990468      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...


✅ Epoch 3 Validation F1: 0.8200

📊 Epoch별 Validation F1 점수:
  1: 0.7953 
  2: 0.8255 ✅ Best
  3: 0.8200 

🔎 Finding best threshold based on validation F1...


Validating for threshold: 100%|██████████| 4340/4340 [00:54<00:00, 79.18it/s]


✅ Best threshold found: 0.53 (F1=0.8296)

🚀 Generating confidence scores with best threshold...


Predicting: 100%|██████████| 18591/18591 [03:47<00:00, 81.62it/s]



🎯 Confidence CSV saved as: confidence_pythia_lora_supcon.csv


In [ ]:
# ====================================================
# 0️⃣ 런타임 최상단에서 환경 안정화
# ====================================================
!pip install protobuf==3.20.3 --quiet
!pip install transformers accelerate --quiet

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from tqdm import tqdm

# ====================================================
# 1️⃣ 설정
# ====================================================
model_name = "microsoft/deberta-v3-small"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 3
batch_size = 8
lr = 2e-5
max_len = 512

# ====================================================
# 2️⃣ 데이터 로드
# ====================================================
train_path = "/kaggle/input/hallym-nlp-competetion-2025/Train.csv"
test_path = "/kaggle/input/hallym-nlp-competetion-2025/Test.csv"
sample_path = "/kaggle/input/hallym-nlp-competetion-2025/sample_submission.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
sample_df = pd.read_csv(sample_path)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# title + text 결합
train_df["text"] = train_df["title"].fillna("") + " " + train_df["text"].fillna("")
test_df["text"] = test_df["title"].fillna("") + " " + test_df["text"].fillna("")

# 512 토큰 초과 데이터 제거
train_df["token_length"] = train_df["text"].apply(lambda x: len(tokenizer.encode(x, truncation=False)))
train_df = train_df[train_df["token_length"] <= max_len].reset_index(drop=True)

print(f"✅ Filtered train size: {len(train_df)}")
print(train_df["label"].value_counts(normalize=True).map(lambda x: f"{x*100:.2f}%"))

# ====================================================
# 3️⃣ Dataset 정의
# ====================================================
class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in inputs.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ====================================================
# 4️⃣ 단일 학습/검증 분할
# ====================================================
train_sub, val_sub = train_test_split(train_df, test_size=0.2, stratify=train_df["label"], random_state=42)

train_loader = DataLoader(NewsDataset(train_sub, tokenizer), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(NewsDataset(val_sub, tokenizer), batch_size=batch_size)

# ====================================================
# 5️⃣ 모델 준비
# ====================================================
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
optimizer = AdamW(model.parameters(), lr=lr)

best_f1, best_thr = 0, 0.5
save_path = "/kaggle/working/best_model.pt"

# ====================================================
# 6️⃣ 학습 루프
# ====================================================
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        optimizer.zero_grad()
        inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(device)
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Train Loss: {avg_loss:.4f}")

    # ----------------- Validation -----------------
    model.eval()
    probs, trues = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Eval]"):
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            labels = batch["labels"].to(device)
            logits = model(**inputs).logits
            prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            probs.extend(prob)
            trues.extend(labels.cpu().numpy())

    probs, trues = np.array(probs), np.array(trues)

    # ----------------- Threshold tuning -----------------
    best_f1_epoch, best_thr_epoch = 0, 0.5
    for thr in np.linspace(0.3, 0.7, 41):
        preds = (probs > thr).astype(int)
        f1 = f1_score(trues, preds)
        if f1 > best_f1_epoch:
            best_f1_epoch, best_thr_epoch = f1, thr

    print(f"Epoch {epoch+1} | Val F1={best_f1_epoch:.4f} | Thr={best_thr_epoch:.3f}")

    if best_f1_epoch > best_f1:
        best_f1, best_thr = best_f1_epoch, best_thr_epoch
        torch.save(model.state_dict(), save_path)
        print(f"✅ Best model saved (F1={best_f1:.4f}, Thr={best_thr:.3f})")

# ====================================================
# 7️⃣ Test 예측 + 확신도 계산
# ====================================================
model.load_state_dict(torch.load(save_path))
model.eval()

all_probs = []
for i in range(0, len(test_df), batch_size):
    texts = test_df["text"].iloc[i:i+batch_size].tolist()
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=max_len
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(prob)

all_probs = np.array(all_probs)

# ----------------- 확신도 매핑 -----------------
confidences = np.zeros_like(all_probs)
for i, p in enumerate(all_probs):
    if p < best_thr:
        confidences[i] = (p / best_thr) * 0.5
    else:
        confidences[i] = 0.5 + ((p - best_thr) / (1 - best_thr)) * 0.5

pred_labels = (all_probs > best_thr).astype(int)

# ====================================================
# 8️⃣ 결과 저장 (3개 파일)
# ====================================================
# (1) 이진 분류 결과
submission_df = sample_df.copy()
submission_df["label"] = pred_labels
submission_path = "/kaggle/working/submission_deberta.csv"
submission_df.to_csv(submission_path, index=False)

# (2) 확신도 결과
confidence_df = sample_df.copy()
confidence_df["confidence"] = confidences
confidence_path = "/kaggle/working/confidence_deberta.csv"
confidence_df.to_csv(confidence_path, index=False)

# (3) 모델 가중치 저장
model_path = "/kaggle/working/best_model.pt"  # 이미 저장됨

print("\n✅ 저장 완료!")
print(f"- 예측 결과 파일: {submission_path}")
print(f"- 확신도 파일:    {confidence_path}")
print(f"- 모델 가중치:    {model_path}")

submission_df.head()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, bu

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
